# nGPT против GPT на Kaggle

Проверяем главный вывод статьи nGPT (arXiv:2410.01131): **nGPT доходит до того же val loss, что обычный трансформер, за меньшее число токенов**.

Что делает ноутбук:
1. Скачивает код из GitHub и токенизирует часть OpenWebText (только в первый раз, ~30–60 мин).
2. Делает lr-свип обеих моделей на 100M токенов, потом обучает лучшие варианты на 300M и 1B токенов.
3. Строит отчёт: val loss от числа токенов и во сколько раз nGPT быстрее.

**Настройки справа (Session options):**
- Accelerator: **GPU T4 x2** (две GPU — GPT и nGPT учатся одновременно);
- Internet: **On** (нужна подтверждённая по телефону учётная запись).

**Как запускать:** `Save Version` → **Save & Run All (Commit)**. Ноутбук работает до ~11.5 ч без открытого браузера.
Если за одну сессию не успело (весь план по грубой оценке 12–25 ч на T4 x2, точнее покажет замер скорости в шаге 3):
1. открой готовую версию → `Output`;
2. в редакторе: `Add Input` → `Your Work` → этот же ноутбук (его последняя версия);
3. снова `Save & Run All`. Ноутбук сам найдёт данные и прогоны во входе и продолжит с того же шага.

In [ ]:
# ===== Настройки =====
SESSION_HOURS = 11.5   # Kaggle останавливает сессию через 12 ч; оставляем запас на сохранение
BUDGET_SCALE = 1.0     # 0.5 — все бюджеты вдвое меньше (быстрее, но ускорение видно хуже)
TRAIN_TOKENS = "1.5e9" # сколько токенов OpenWebText подготовить (3 ГБ на диске)
REPO = "https://github.com/NordManul/Project_Transformers.git"

import glob, json, os, shutil, subprocess, time
T0 = time.time()
WORK = "/kaggle/working"
CODE = f"{WORK}/Project_Transformers"
DATA = f"{WORK}/data/owt"
OUT = f"{WORK}/runs/ngpt_vs_gpt"

if not os.path.exists(CODE):
    subprocess.run(["git", "clone", "--depth", "1", REPO, CODE], check=True)
else:
    subprocess.run(["git", "-C", CODE, "pull", "--ff-only"], check=False)
os.chdir(CODE)
import torch
print("torch", torch.__version__, "| GPU:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

## 1. Данные

Ищем готовые `train.bin`/`val.bin` во входах (прошлая версия ноутбука). Если нет — токенизируем OpenWebText.
Данные копируются в `/kaggle/working`, чтобы каждая версия ноутбука была самодостаточной.

In [ ]:
found = sorted(glob.glob("/kaggle/input/**/data/owt/train.bin", recursive=True))
if os.path.exists(f"{DATA}/train.bin"):
    print("Данные уже на месте:", DATA)
elif found:
    src = os.path.dirname(found[0])
    print("Копирую данные из входа:", src)
    shutil.copytree(src, DATA, dirs_exist_ok=True)
else:
    !python tokenize_dataset.py --dataset openwebtext --train_tokens {TRAIN_TOKENS} --val_tokens 1e7 --out {DATA}
print(open(f"{DATA}/meta.json").read())

## 2. Прогоны прошлых сессий

Если во входе есть прошлая версия — берём оттуда папку эксперимента с наибольшим числом законченных прогонов.

In [ ]:
def finished_runs(exp_dir):
    n = 0
    for m in glob.glob(f"{exp_dir}/runs/*/metrics.jsonl"):
        n += '"type": "final"' in open(m, encoding="utf-8").read()
    return n

prev = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/runs/ngpt_vs_gpt/plan.json", recursive=True)]
if prev and not os.path.exists(f"{OUT}/plan.json"):
    best = max(prev, key=finished_runs)
    print(f"Продолжаю эксперимент из {best} (законченных прогонов: {finished_runs(best)})")
    shutil.copytree(best, OUT, dirs_exist_ok=True)
elif os.path.exists(f"{OUT}/plan.json"):
    print("Эксперимент уже в рабочей папке:", OUT)
else:
    print("Новый эксперимент")

## 3. Замер скорости (только в первый раз)

Пара десятков шагов каждой модели: токены в секунду, память и сколько часов займёт весь план.

In [ ]:
if not os.path.exists(f"{OUT}/plan.json"):
    !python ngpt_vs_gpt.py bench --data {DATA} --out {OUT} --budget_scale {BUDGET_SCALE}

## 4. Обучение

Работает, пока не кончится время сессии; перед лимитом каждый прогон сохраняет чекпойнт.
Логи каждого прогона: `runs/ngpt_vs_gpt/runs/<имя>/train.log`.

In [ ]:
HOURS = f"{SESSION_HOURS - (time.time() - T0) / 3600 - 0.3:.2f}"   # 0.3 ч — на отчёт и сохранение версии
print("Часов на обучение:", HOURS)
!python ngpt_vs_gpt.py run --data {DATA} --out {OUT} --budget_scale {BUDGET_SCALE} --hours {HOURS}

## 5. Отчёт

In [ ]:
!python ngpt_vs_gpt.py report --out {OUT} > /dev/null
from IPython.display import Image, Markdown, display
display(Markdown(open(f"{OUT}/report/report.md", encoding="utf-8").read()))
for name in ("ngpt_vs_gpt.png", "details/loss.png", "details/architecture_params.png"):
    path = f"{OUT}/report/{name}"
    if os.path.exists(path):
        display(Image(path))